# Expérience 1 — Un signal audio artificiel

But : comprendre ce que sont les **échantillons** et la **forme d'onde** avant de passer au spectrogramme.

⚠️ Le signal créé ici est une sinusoïde à 440 Hz (la note *la*). Ce n'est **pas** une voix humaine, et ce notebook ne fait **ni STT ni TTS** pour l'instant.

## Cellule 1 — Imports et paramètres

- `numpy` : calcul sur des tableaux de nombres (notre signal sera un tableau).
- `matplotlib` : tracer des courbes.
- `sample_rate` : la **fréquence d'échantillonnage**, c'est-à-dire le nombre de mesures prises par seconde.
- `duration` : la durée du signal, en secondes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Paramètres du signal
sample_rate = 16000  # 16 000 échantillons par seconde
duration = 3         # durée en secondes

## Cellule 2 — Création du signal

- `np.arange(sample_rate * duration)` produit les numéros d'échantillons `0, 1, 2, …, 47999`.
- En les divisant par `sample_rate`, on obtient l'**instant** (en secondes) de chaque mesure : `0, 1/16000, 2/16000, …`
- `np.sin(2 * np.pi * frequency * t)` calcule, à chacun de ces instants, la valeur d'une onde qui fait `frequency` = 440 oscillations par seconde.

In [ ]:
# Création des instants de mesure
t = np.arange(sample_rate * duration) / sample_rate

# Création d'un signal sinusoïdal artificiel à 440 Hz
frequency = 440
audio = np.sin(2 * np.pi * frequency * t)

## Cellule 3 — Informations sur le signal

À vérifier dans la sortie :
1. le nombre d'échantillons ;
2. la durée calculée ;
3. les 10 premières valeurs (la première valeur et le signe des suivantes).

In [ ]:
print("Fréquence d'échantillonnage :", sample_rate, "Hz")
print("Nombre d'échantillons :", len(audio))
print("Durée :", len(audio) / sample_rate, "secondes")
print("Premières valeurs :", audio[:10])

## Cellule 4 — Visualisation d'un extrait

On ne trace que les 200 premiers échantillons : avec les 48 000, les oscillations seraient trop serrées pour être visibles.

À observer : l'axe horizontal (temps), l'axe vertical (amplitude) et le **nombre de cycles complets** visibles.

In [ ]:
plt.plot(t[:200], audio[:200])
plt.xlabel("Temps (secondes)")
plt.ylabel("Amplitude")
plt.title("Extrait d'un signal audio artificiel")
plt.grid()
plt.show()

# Expérience 2 — Des fréquences contenues dans un signal (FFT)

La forme d'onde montre **comment l'amplitude varie dans le temps**, mais ne dit pas directement **quelles fréquences** sont présentes.

La **FFT** (*Fast Fourier Transform*, transformée de Fourier rapide) répond à cette question : elle décompose un morceau de signal en une somme de sinusoïdes, et indique l'**intensité** de chaque fréquence.

Analogie : la forme d'onde, c'est le son d'un accord ; la FFT, c'est la liste des notes qui composent cet accord.

C'est la brique de base du **STFT** (et donc du spectrogramme) : le STFT, c'est simplement des FFT faites sur de petites fenêtres successives.

## Cellule 5 — FFT sur une seule fenêtre de 25 ms

- On prend **400 échantillons** = 400 / 16 000 = **25 ms** (taille de fenêtre courante en parole).
- `np.fft.rfft` calcule la FFT (version pour signaux réels : seulement les fréquences positives).
- `np.abs(...)` donne la **magnitude** : l'intensité de chaque fréquence.
- `np.fft.rfftfreq(n, 1/sample_rate)` donne la fréquence (en Hz) correspondant à chaque case du résultat.

À observer : la position du pic, et l'écart entre deux fréquences consécutives (la **résolution fréquentielle**).

In [ ]:
window_size = 400                        # 25 ms à 16 kHz
window = audio[:window_size]

spectrum = np.abs(np.fft.rfft(window))   # magnitude de chaque fréquence
freqs = np.fft.rfftfreq(window_size, 1 / sample_rate)

print("Nombre de fréquences analysées :", len(freqs))
print("Écart entre deux fréquences :", freqs[1] - freqs[0], "Hz")
print("Fréquence la plus forte :", freqs[np.argmax(spectrum)], "Hz")

plt.plot(freqs, spectrum)
plt.xlim(0, 2000)
plt.xlabel("Fréquence (Hz)")
plt.ylabel("Magnitude")
plt.title("Spectre d'une fenêtre de 25 ms (signal à 440 Hz)")
plt.grid()
plt.show()

## Cellule 6 — Un signal composé de deux fréquences

On additionne une sinusoïde à 440 Hz et une autre à 1000 Hz, deux fois plus faible (amplitude 0,5).

À observer : sur la forme d'onde, c'est difficile de « voir » les deux fréquences ; sur le spectre, elles apparaissent séparément.

In [ ]:
mix = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 1000 * t)

mix_spectrum = np.abs(np.fft.rfft(mix[:window_size]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(t[:window_size], mix[:window_size])
ax1.set_xlabel("Temps (secondes)")
ax1.set_ylabel("Amplitude")
ax1.set_title("Forme d'onde : 440 Hz + 1000 Hz")
ax1.grid()

ax2.plot(freqs, mix_spectrum)
ax2.set_xlim(0, 2000)
ax2.set_xlabel("Fréquence (Hz)")
ax2.set_ylabel("Magnitude")
ax2.set_title("Spectre de la même fenêtre")
ax2.grid()
plt.show()

# Expérience 3 — Du STFT au spectrogramme

Un signal dont le contenu **change dans le temps** :
- de 0 à 1 s : 440 Hz seul ;
- de 1 à 2 s : 1000 Hz seul ;
- de 2 à 3 s : 440 Hz + 1000 Hz ensemble.

Une seule FFT sur tout le signal verrait 440 et 1000 Hz, mais ne dirait pas **quand**. Le STFT répond à cette question.

In [ ]:
segment = sample_rate  # 1 seconde = 16 000 échantillons
t1 = t[:segment]

part1 = np.sin(2 * np.pi * 440 * t1)
part2 = np.sin(2 * np.pi * 1000 * t1)
part3 = np.sin(2 * np.pi * 440 * t1) + 0.5 * np.sin(2 * np.pi * 1000 * t1)

changing = np.concatenate([part1, part2, part3])
print("Nombre d'échantillons :", len(changing))

## Cellule 8 — Le STFT écrit à la main

- `window_size` : taille de chaque fenêtre (400 échantillons = 25 ms).
- `hop` : de combien on décale la fenêtre à chaque pas (160 échantillons = 10 ms). Les fenêtres se **chevauchent**.
- Pour chaque fenêtre, on calcule une FFT, et on range le résultat comme une ligne de la matrice.

À observer : la **forme** (`shape`) de la matrice. Essaie de prédire le nombre de lignes avant d'exécuter.

In [ ]:
def stft_magnitude(signal, window_size, hop):
    frames = []
    for start in range(0, len(signal) - window_size + 1, hop):
        window = signal[start:start + window_size]
        frames.append(np.abs(np.fft.rfft(window)))
    return np.array(frames)  # forme : (nombre de fenêtres, nombre de fréquences)

hop = 160  # 10 ms
spec = stft_magnitude(changing, window_size, hop)
print("Forme du spectrogramme :", spec.shape)

## Cellule 9 — Afficher le spectrogramme

- Axe horizontal : le **temps** (une colonne par fenêtre).
- Axe vertical : la **fréquence**.
- Couleur : la **magnitude** (claire = forte énergie).

`.T` transpose la matrice pour mettre le temps en horizontal, comme on le fait habituellement.

In [ ]:
duration_total = len(changing) / sample_rate
plt.figure(figsize=(10, 4))
plt.imshow(spec.T, origin="lower", aspect="auto",
           extent=[0, duration_total, 0, sample_rate / 2])
plt.ylim(0, 2000)
plt.xlabel("Temps (secondes)")
plt.ylabel("Fréquence (Hz)")
plt.title(f"Spectrogramme (fenêtre = {window_size} échantillons)")
plt.colorbar(label="Magnitude")
plt.show()

## Cellule 10 — Le compromis temps / fréquence

Même signal, trois tailles de fenêtre : 100 (6,25 ms), 400 (25 ms) et 1600 (100 ms).

À observer pour chaque taille :
- la **netteté des lignes horizontales** (précision en fréquence) ;
- la **netteté des transitions** à 1 s et 2 s (précision dans le temps).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, size in zip(axes, [100, 400, 1600]):
    s = stft_magnitude(changing, size, hop)
    ax.imshow(s.T, origin="lower", aspect="auto",
              extent=[0, duration_total, 0, sample_rate / 2])
    ax.set_ylim(0, 2000)
    ax.set_title(f"Fenêtre = {size} ({size / sample_rate * 1000:.2f} ms) — résolution {sample_rate / size:.0f} Hz")
    ax.set_xlabel("Temps (s)")
axes[0].set_ylabel("Fréquence (Hz)")
plt.tight_layout()
plt.show()